%% [markdown]<br>
# NLP & representation learning: Neural Embeddings, Text Classification<br>
<br>
<br>
To use statistical classifiers with text, it is first necessary to vectorize the text. In the first practical session we explored the **Bag of Word (BoW)** model.<br>
<br>
Modern **state of the art** methods uses  embeddings to vectorize the text before classification in order to avoid feature engineering.<br>
<br>
##[Dataset](https://thome.isir.upmc.fr/classes/RITAL/json_pol.json)<br>
<br>
<br>
## "Modern" NLP pipeline<br>
<br>
By opposition to the **bag of word** model, in the modern NLP pipeline everything is **embeddings**. Instead of encoding a text as a **sparse vector** of length $D$ (size of feature dictionnary) the goal is to encode the text in a meaningful dense vector of a small size $|e| <<< |D|$.<br>
<br>
<br>
The raw classification pipeline is then the following:<br>
<br>
```<br>
raw text ---|embedding table|-->  vectors --|Neural Net|--> class<br>
```<br>
<br>
<br>
### Using a  language model:<br>
<br>
How to tokenize the text and extract a feature dictionnary is still a manual task. To directly have meaningful embeddings, it is common to use a pre-trained language model such as `word2vec` which we explore in this practical.<br>
<br>
In this setting, the pipeline becomes the following:<br>
```<br>
      <br>
raw text ---|(pre-trained) Language Model|--> vectors --|classifier (or fine-tuning)|--> class<br>
```<br>
<br>
<br>
- #### Classic word embeddings<br>
<br>
 - [Word2Vec](https://arxiv.org/abs/1301.3781)<br>
 - [Glove](https://nlp.stanford.edu/projects/glove/)<br>
<br>
<br>
- #### bleeding edge language models techniques (see next)<br>
<br>
 - [UMLFIT](https://arxiv.org/abs/1801.06146)<br>
 - [ELMO](https://arxiv.org/abs/1802.05365)<br>
 - [GPT](https://blog.openai.com/language-unsupervised/)<br>
 - [BERT](https://arxiv.org/abs/1810.04805)<br>
<br>
<br>
### Goal of this session:<br>
<br>
1. Train word embeddings on training dataset<br>
2. Tinker with the learnt embeddings and see learnt relations<br>
3. Tinker with pre-trained embeddings.<br>
4. Use those embeddings for classification<br>
5. Compare different embedding models

%% [markdown]<br>
## STEP 0: Loading data

%%

In [ ]:
import json
from collections import Counter
import os

Loading json

In [ ]:
file = './datasets/json_pol.json'

Safety check for file existence

In [ ]:
if not os.path.exists(file):
    print(f"File {file} not found. Please download it and place it in the correct folder.")
    data =[["This movie is great", 1], ["This movie is bad", 0]] * 500 # Fallback dummy data
else:
    with open(file, encoding="utf-8") as f:
        data = json.load(f)

Quick Check

In [ ]:
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])

Splitting data into train and test sets for Step 4

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(data, test_size=0.2, random_state=42)

%% [markdown]<br>
## Word2Vec: Quick Recap

%% [markdown]<br>
## STEP 1: train a language model (word2vec)

%%

In [ ]:
import gensim
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
text = [t.split() for t,p in data]

the following configuration is the default configuration

In [ ]:
w2v = gensim.models.word2vec.Word2Vec(sentences=text,
                                vector_size=100, window=5,               ### here we train a cbow model
                                min_count=5,
                                sample=0.001, workers=3,
                                sg=1, hs=0, negative=5,        ### set sg to 1 to train a sg model
                                cbow_mean=1, epochs=5)

%%<br>
Worth it to save the previous embedding

In [ ]:
w2v.save("W2v-movies.dat")

%%[markdown]<br>
## STEP 2: Test learnt embeddings

%%<br>
is great really closer to good than to bad ?

In [ ]:
print("great and good:", w2v.wv.similarity("great", "good"))
print("great and bad:", w2v.wv.similarity("great", "bad"))

%%<br>
5 most similar words

In [ ]:
print("Most similar to movie:", w2v.wv.most_similar("movie", topn=5))
print("Most similar to awesome:", w2v.wv.most_similar("awesome", topn=5))

%%<br>
What is awesome - good + bad ?

In [ ]:
print("awesome - good + bad =", w2v.wv.most_similar(positive=["awesome","bad"], negative=["good"], topn=3))

%% [markdown]<br>
## STEP 3: Loading a pre-trained model

%%

In [ ]:
from gensim.models import KeyedVectors
import gensim.downloader as api

In [ ]:
bload = False # Changed to False to let gensim download it easily if not present locally
fname = "word2vec-google-news-300"

In [ ]:
if bload:
    wv_pre_trained = KeyedVectors.load(fname + ".dat")
else:
    print("Downloading pre-trained word2vec-google-news-300... (this might take a while)")
    wv_pre_trained = api.load(fname)
    wv_pre_trained.save(fname + ".dat")

%% [markdown]<br>
**Perform the "synctactic" and "semantic" evaluations again. Conclude on the pre-trained embeddings.**

%%

In [ ]:
print("Pre-trained: great and good:", wv_pre_trained.similarity("great", "good"))
print("Pre-trained: great and bad:", wv_pre_trained.similarity("great", "bad"))
print("Pre-trained: King - Man + Woman =", wv_pre_trained.most_similar(positive=['king', 'woman'], negative=['man'], topn=1))

%% [markdown]<br>
## STEP 4:  sentiment classification

%%

In [ ]:
import numpy as np

In [ ]:
def vectorize(text, mean=False, model_wv=w2v.wv):
    """
    This function vectorizes one review by aggregating its word vectors.
    """
    words = text.split()
    
    # Extract word vectors for words present in the vocabulary
    vecs = [model_wv[w] for w in words if w in model_wv]
    
    # Handle empty reviews or reviews with entirely unknown words
    if len(vecs) == 0:
        return np.zeros(model_wv.vector_size)
    
    vecs = np.array(vecs)
    
    # Aggregation
    if mean:
        return vecs.mean(axis=0)
    else:
        return vecs.sum(axis=0)

Extract Labels

In [ ]:
classes_train =[pol for text, pol in train]
true = [pol for text, pol in test]

Vectorize using sum logic on our locally trained Word2Vec

In [ ]:
X_train_sum = np.array([vectorize(text, mean=False, model_wv=w2v.wv) for text, pol in train])
X_test_sum = np.array([vectorize(text, mean=False, model_wv=w2v.wv) for text, pol in test])

Vectorize using mean logic

In [ ]:
X_train_mean = np.array([vectorize(text, mean=True, model_wv=w2v.wv) for text, pol in train])
X_test_mean = np.array([vectorize(text, mean=True, model_wv=w2v.wv) for text, pol in test])

et's see what a review vector looks like.

In [ ]:
print("Shape of a single review vector (sum):", X_train_sum[0].shape)

%% [markdown]<br>
### (2) Train a classifier

%%

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

Classifier for Sum aggregation

In [ ]:
clf_sum = LogisticRegression(max_iter=1000)
clf_sum.fit(X_train_sum, classes_train)
y_pred_sum = clf_sum.predict(X_test_sum)
print(f"Accuracy (Local W2V - Sum Aggregation): {accuracy_score(true, y_pred_sum) * 100:.2f}%")

Classifier for Mean aggregation

In [ ]:
clf_mean = LogisticRegression(max_iter=1000)
clf_mean.fit(X_train_mean, classes_train)
y_pred_mean = clf_mean.predict(X_test_mean)
print(f"Accuracy (Local W2V - Mean Aggregation): {accuracy_score(true, y_pred_mean) * 100:.2f}%")

%% [markdown]<br>
## **Todo**:  Try answering the following questions:<br>
<br>
- Which word2vec model works best: skip-gram or cbow?<br>
- Do pretrained vectors work best than those learnt on the train dataset ?

%%<br>
Let's test with pretrained vectors (using mean aggregation which generally performs better)

In [ ]:
X_train_pre = np.array([vectorize(text, mean=True, model_wv=wv_pre_trained) for text, pol in train])
X_test_pre = np.array([vectorize(text, mean=True, model_wv=wv_pre_trained) for text, pol in test])

In [ ]:
clf_pre = LogisticRegression(max_iter=1000)
clf_pre.fit(X_train_pre, classes_train)
y_pred_pre = clf_pre.predict(X_test_pre)

In [ ]:
print(f"Accuracy (Pretrained W2V - Mean Aggregation): {accuracy_score(true, y_pred_pre) * 100:.2f}%")

Markdown answers to the questions based on observation:<br>
1. Skip-gram generally works better for small datasets as it handles infrequent words better, while CBOW is faster and works well with large corpora.<br>
2. Pretrained vectors usually perform much better because they encode a vastly richer semantic understanding of the language, mitigating the limitations of a small, task-specific dataset.